# 🧱 DCGAN - Bricks Data

In this notebook, we'll walk through the steps required to train your own DCGAN on the bricks dataset

In [ ]:
%load_ext autoreload
%autoreload 2
import os
from collections import defaultdict
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path
from PIL import Image

import torch
from torch import nn
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
from safetensors.torch import load_file
from transformers import Trainer, TrainingArguments, TrainerCallback
from transformers.trainer_utils import get_last_checkpoint

import sys
from pathlib import Path

# Make the repo root importable so `notebooks.utils_pytorch` can be found
REPO_ROOT = Path.cwd().resolve().parents[2]
sys.path.insert(0, str(REPO_ROOT))

from notebooks.utils_pytorch import display

# The Hugging Face Trainer picks the best available device
# automatically, so we only report what it will find
print(f"CUDA available: {torch.cuda.is_available()}")
print(f"MPS available: {torch.backends.mps.is_available()}")

## 0. Parameters <a name="parameters"></a>

In [ ]:
IMAGE_SIZE = 64
CHANNELS = 1
BATCH_SIZE = 128
Z_DIM = 100
EPOCHS = 300
LOAD_MODEL = False
ADAM_BETA_1 = 0.5
ADAM_BETA_2 = 0.999
LEARNING_RATE = 0.0002
NOISE_PARAM = 0.1

## 1. Prepare the data <a name="prepare"></a>

In [ ]:
class ImageDataset(Dataset):
    """
    All image files found under `folder` (searched recursively, in sorted order),
    resized to `image_size` - the PyTorch counterpart of Keras'
    image_dataset_from_directory. Images are uint8 tensors of shape (C, H, W).
    """

    def __init__(self, folder, image_size, color_mode="rgb", labels=None):
        exts = {".jpg", ".jpeg", ".png", ".bmp", ".gif"}
        self.paths = sorted(p for p in Path(folder).rglob("*") if p.suffix.lower() in exts)
        self.mode = {"grayscale": "L", "rgb": "RGB"}[color_mode]
        self.resize = transforms.Resize(
            image_size, interpolation=transforms.InterpolationMode.BILINEAR
        )
        self.labels = labels
        self.transform = None  # optional function applied to each image
        self.target_transform = None  # optional function applied to each label
        print(f"Found {len(self.paths)} files.")

    def __len__(self):
        return len(self.paths)

    def __getitem__(self, idx):
        img = Image.open(self.paths[idx]).convert(self.mode)
        img = transforms.functional.pil_to_tensor(self.resize(img))
        if self.transform:
            img = self.transform(img)
        # The Trainer passes each example to the model as keyword arguments,
        # so we return a dict whose keys match the model's forward()
        if self.labels is None:
            return {"pixel_values": img}
        label = self.labels[idx]
        if self.target_transform:
            label = self.target_transform(label)
        return {"pixel_values": img, "labels": label}


train_data = ImageDataset(
    "../../../data/lego-brick-images/dataset/",
    image_size=(IMAGE_SIZE, IMAGE_SIZE),
    color_mode="grayscale",
)

In [ ]:
def preprocess(img):
    """
    Normalize the images to [-1, 1]
    """
    img = (img.float() - 127.5) / 127.5
    return img


# Like Keras' .map(): preprocess is applied to every image as it is loaded.
# No DataLoader needed: the Trainer batches and shuffles the dataset itself.
train_data.transform = preprocess
train = train_data

In [ ]:
# A DataLoader just to look at one batch (the Trainer makes its own for training)
train_sample = next(iter(DataLoader(train, batch_size=BATCH_SIZE, shuffle=True)))["pixel_values"]

In [ ]:
display(train_sample)

## 2. Build the GAN <a name="build"></a>

In [ ]:
# PyTorch images are channels-first: (batch, channels, height, width).
# padding=1 with kernel 4 / stride 2 halves the size, like Keras' padding="same".
discriminator = nn.Sequential(
    nn.Conv2d(CHANNELS, 64, kernel_size=4, stride=2, padding=1, bias=False),
    nn.LeakyReLU(0.2),
    nn.Dropout(0.3),
    nn.Conv2d(64, 128, kernel_size=4, stride=2, padding=1, bias=False),
    # PyTorch momentum = 1 - Keras momentum (Keras 0.9 -> PyTorch 0.1)
    nn.BatchNorm2d(128, momentum=0.1),
    nn.LeakyReLU(0.2),
    nn.Dropout(0.3),
    nn.Conv2d(128, 256, kernel_size=4, stride=2, padding=1, bias=False),
    nn.BatchNorm2d(256, momentum=0.1),
    nn.LeakyReLU(0.2),
    nn.Dropout(0.3),
    nn.Conv2d(256, 512, kernel_size=4, stride=2, padding=1, bias=False),
    nn.BatchNorm2d(512, momentum=0.1),
    nn.LeakyReLU(0.2),
    nn.Dropout(0.3),
    # No sigmoid here: the discriminator outputs a raw score (logit) and
    # nn.BCEWithLogitsLoss applies the sigmoid inside the loss (more stable)
    nn.Conv2d(512, 1, kernel_size=4, stride=1, padding=0, bias=False),
    nn.Flatten(),
)

print(discriminator)
print(f"Total params: {sum(p.numel() for p in discriminator.parameters()):,}")

In [ ]:
generator = nn.Sequential(
    nn.Unflatten(1, (Z_DIM, 1, 1)),  # (batch, Z_DIM) -> (batch, Z_DIM, 1, 1)
    nn.ConvTranspose2d(Z_DIM, 512, kernel_size=4, stride=1, padding=0, bias=False),
    nn.BatchNorm2d(512, momentum=0.1),
    nn.LeakyReLU(0.2),
    # padding=1 with kernel 4 / stride 2 doubles the size, like Keras' padding="same"
    nn.ConvTranspose2d(512, 256, kernel_size=4, stride=2, padding=1, bias=False),
    nn.BatchNorm2d(256, momentum=0.1),
    nn.LeakyReLU(0.2),
    nn.ConvTranspose2d(256, 128, kernel_size=4, stride=2, padding=1, bias=False),
    nn.BatchNorm2d(128, momentum=0.1),
    nn.LeakyReLU(0.2),
    nn.ConvTranspose2d(128, 64, kernel_size=4, stride=2, padding=1, bias=False),
    nn.BatchNorm2d(64, momentum=0.1),
    nn.LeakyReLU(0.2),
    nn.ConvTranspose2d(64, CHANNELS, kernel_size=4, stride=2, padding=1, bias=False),
    nn.Tanh(),
)

print(generator)
print(f"Total params: {sum(p.numel() for p in generator.parameters()):,}")

In [ ]:
class DCGAN(nn.Module):
    def __init__(self, discriminator, generator, latent_dim):
        super().__init__()
        self.discriminator = discriminator
        self.generator = generator
        self.latent_dim = latent_dim
        # Takes logits; unlike nn.BCELoss it also accepts the noisy labels
        # below, which fall slightly outside [0, 1]
        self.loss_fn = nn.BCEWithLogitsLoss()

    def discriminator_loss(self, pixel_values):
        # Sample random points in the latent space
        batch_size = pixel_values.shape[0]
        random_latent_vectors = torch.randn(batch_size, self.latent_dim, device=pixel_values.device)
        with torch.no_grad():  # the generator is not trained in this step
            generated_images = self.generator(random_latent_vectors)

        real_predictions = self.discriminator(pixel_values)
        fake_predictions = self.discriminator(generated_images)

        real_labels = torch.ones_like(real_predictions)
        real_noisy_labels = real_labels + NOISE_PARAM * torch.rand_like(real_predictions)
        fake_labels = torch.zeros_like(fake_predictions)
        fake_noisy_labels = fake_labels - NOISE_PARAM * torch.rand_like(fake_predictions)

        d_real_loss = self.loss_fn(real_predictions, real_noisy_labels)
        d_fake_loss = self.loss_fn(fake_predictions, fake_noisy_labels)
        d_loss = (d_real_loss + d_fake_loss) / 2.0

        # Metrics (a logit > 0 means the discriminator says "real")
        d_real_acc = (real_predictions > 0).float().mean()
        d_fake_acc = (fake_predictions < 0).float().mean()
        metrics = {
            "d_loss": d_loss.item(),
            "d_real_acc": d_real_acc.item(),
            "d_fake_acc": d_fake_acc.item(),
            "d_acc": ((d_real_acc + d_fake_acc) / 2).item(),
        }
        return d_loss, metrics

    def forward(self, pixel_values):
        # The generator loss - the "loss" the Trainer optimizes. The generator
        # wants the discriminator to call its images real.
        batch_size = pixel_values.shape[0]
        random_latent_vectors = torch.randn(batch_size, self.latent_dim, device=pixel_values.device)
        g_predictions = self.discriminator(self.generator(random_latent_vectors))
        g_loss = self.loss_fn(g_predictions, torch.ones_like(g_predictions))
        return {"loss": g_loss, "g_acc": (g_predictions > 0).float().mean()}


class GANTrainer(Trainer):
    """
    The standard Trainer does one loss / one optimizer per batch. A GAN trains two
    networks in turn, each with its own optimizer, so we override:
      - create_optimizer: one Adam for each network
      - training_step: update the discriminator first, then the generator
      - log: add our metrics to the Trainer's log
    """

    def __init__(self, *args, **kwargs):
        super().__init__(*args, **kwargs)
        self.gan_metrics = defaultdict(list)

    def create_optimizer(self, model=None):
        lr, betas = self.args.learning_rate, (self.args.adam_beta1, self.args.adam_beta2)
        # self.optimizer is the one the Trainer steps for us: the generator's
        self.optimizer = torch.optim.Adam(self.model.generator.parameters(), lr=lr, betas=betas)
        # The discriminator's optimizer we step ourselves in training_step
        self.d_optimizer = torch.optim.Adam(self.model.discriminator.parameters(), lr=lr, betas=betas)
        return self.optimizer

    def training_step(self, model, inputs, num_items_in_batch=None):
        model.train()
        inputs = self._prepare_inputs(inputs)  # move the batch to the device

        # Train the discriminator on real and fake images
        d_loss, metrics = model.discriminator_loss(**inputs)
        self.d_optimizer.zero_grad()
        self.accelerator.backward(d_loss)
        self.d_optimizer.step()

        # Train the generator: we only compute the gradients here; the Trainer
        # then calls self.optimizer.step() and zero_grad(), as for any model
        outputs = model(**inputs)
        self.accelerator.backward(outputs["loss"])

        metrics.update(g_loss=outputs["loss"].item(), g_acc=outputs["g_acc"].item())
        for name, value in metrics.items():
            self.gan_metrics[name].append(value)
        return outputs["loss"].detach()

    def log(self, logs, *args, **kwargs):
        # Replace the Trainer's "loss" (which is just g_loss) by our averaged metrics
        if self.gan_metrics and "loss" in logs:
            logs.pop("loss")
            logs.update({name: float(np.mean(v)) for name, v in self.gan_metrics.items()})
            self.gan_metrics.clear()
        super().log(logs, *args, **kwargs)

In [ ]:
# Create a DCGAN
dcgan = DCGAN(
    discriminator=discriminator, generator=generator, latent_dim=Z_DIM
)

In [ ]:
if LOAD_MODEL:
    # The Trainer saves each checkpoint as ./checkpoint/checkpoint-<step>/model.safetensors
    checkpoint_dir = get_last_checkpoint("./checkpoint")
    dcgan.load_state_dict(load_file(os.path.join(checkpoint_dir, "model.safetensors")))

## 3. Train the GAN <a name="train"></a>

In [ ]:
# TrainingArguments holds the optimizer and training settings
# (GANTrainer.create_optimizer builds one Adam per network from them)
training_args = TrainingArguments(
    output_dir="./checkpoint",
    learning_rate=LEARNING_RATE,
    adam_beta1=ADAM_BETA_1,
    adam_beta2=ADAM_BETA_2,
    lr_scheduler_type="constant",  # Keras keeps the learning rate fixed
    max_grad_norm=0.0,  # the Trainer clips gradients by default; Keras doesn't
    per_device_train_batch_size=BATCH_SIZE,
    num_train_epochs=EPOCHS,
    logging_strategy="epoch",
    save_strategy="epoch",
    save_only_model=True,  # weights only, like save_weights_only=True
    save_total_limit=1,  # keep just the latest checkpoint
    report_to="tensorboard",
    remove_unused_columns=False,  # our dataset keys are already what the model needs
)

In [ ]:
# TensorBoard logs go to ./logs (set with an environment variable in transformers 5)
os.environ["TENSORBOARD_LOGGING_DIR"] = "./logs"


class ImageGenerator(TrainerCallback):
    def __init__(self, num_img, latent_dim):
        self.num_img = num_img
        self.latent_dim = latent_dim

    def on_epoch_end(self, args, state, control, model=None, **kwargs):
        epoch = round(state.epoch) - 1  # state.epoch counts completed epochs
        generator = model.generator
        generator.eval()  # BatchNorm uses its running statistics, like Keras' training=False
        with torch.no_grad():
            random_latent_vectors = torch.randn(self.num_img, self.latent_dim, device=args.device)
            generated_images = generator(random_latent_vectors)
        generator.train()
        generated_images = generated_images * 127.5 + 127.5
        display(
            generated_images,
            save_to="./output/generated_img_%03d.png" % (epoch),
        )

In [ ]:
trainer = GANTrainer(
    model=dcgan,
    args=training_args,
    train_dataset=train,
    callbacks=[ImageGenerator(num_img=10, latent_dim=Z_DIM)],
)
trainer.train()

In [ ]:
# Save the final models
torch.save(generator.state_dict(), "./models/generator.pt")
torch.save(discriminator.state_dict(), "./models/discriminator.pt")

## 3. Generate new images <a name="decode"></a>

In [ ]:
# Sample some points in the latent space, from the standard normal distribution
grid_width, grid_height = (10, 3)
z_sample = np.random.normal(size=(grid_width * grid_height, Z_DIM))

In [ ]:
# Decode the sampled points
generator.eval()  # BatchNorm in inference mode, like Keras' predict()
with torch.no_grad():
    z = torch.tensor(z_sample, dtype=torch.float32, device=trainer.args.device)
    reconstructions = generator(z).cpu().numpy()  # (N, 1, H, W)

In [ ]:
# Draw a plot of decoded images
fig = plt.figure(figsize=(18, 5))
fig.subplots_adjust(hspace=0.4, wspace=0.4)

# Output the grid of faces
for i in range(grid_width * grid_height):
    ax = fig.add_subplot(grid_height, grid_width, i + 1)
    ax.axis("off")
    ax.imshow(reconstructions[i, 0], cmap="Greys")  # channel 0 of the grayscale image

In [ ]:
def compare_images(img1, img2):
    return np.mean(np.abs(img1 - img2))

In [ ]:
all_data = []
for i in DataLoader(train, batch_size=BATCH_SIZE):
    all_data.extend(i["pixel_values"].numpy())
all_data = np.array(all_data)

In [ ]:
r, c = 3, 5
fig, axs = plt.subplots(r, c, figsize=(10, 6))
fig.suptitle("Generated images", fontsize=20)

noise = np.random.normal(size=(r * c, Z_DIM))
generator.eval()
with torch.no_grad():
    gen_imgs = generator(torch.tensor(noise, dtype=torch.float32, device=trainer.args.device)).cpu().numpy()

cnt = 0
for i in range(r):
    for j in range(c):
        axs[i, j].imshow(gen_imgs[cnt, 0], cmap="gray_r")
        axs[i, j].axis("off")
        cnt += 1

plt.show()

In [ ]:
fig, axs = plt.subplots(r, c, figsize=(10, 6))
fig.suptitle("Closest images in the training set", fontsize=20)

cnt = 0
for i in range(r):
    for j in range(c):
        c_diff = 99999
        c_img = None
        for k_idx, k in enumerate(all_data):
            diff = compare_images(gen_imgs[cnt], k)
            if diff < c_diff:
                c_img = np.copy(k)
                c_diff = diff
        axs[i, j].imshow(c_img[0], cmap="gray_r")
        axs[i, j].axis("off")
        cnt += 1

plt.show()